# Urban green areas — vegetation index pipeline

Computes a vegetation index (e.g. NDVI) from aerial RGBI orthophotos and builds a city-wide mosaic, in three steps:

1. **Crop** — clip every RGBI tile to the municipality boundary.
2. **Evaluate indices** — compute the vegetation index per cropped tile.
3. **Mosaic** — merge all per-tile index rasters into a single city-wide GeoTIFF.

## Setup & Imports

In [1]:
#%pip install gdal-ecw

In [2]:
#%pip install geopandas

In [3]:
import os
import glob
import numpy as np
import tifffile as tiff
import gc
from osgeo import gdal, osr
import rasterio
from rasterio.mask import mask
from rasterio.merge import merge
import geopandas as gpd
from pathlib import Path

In [4]:
print(gdal.__version__)
#gdal.SieveFilter()

3.4.3


## Configuration

Set here which city/area to process and where its input data lives.

- `procom`: the ISTAT municipality code to process. **It must already be a key of `city_dict` below** (add a new `'code': 'Name'` entry to the dictionary if it is missing, otherwise `city_dict[procom]` will raise a `KeyError`).
- `flight_year`: the year of the aerial survey, used both to build folder/file names and to tag output files.
- `indices`: which vegetation index/indices to compute (see the `evaluate_index()` function further down).

Expected input folder structure, under `base_path`:
```
input/<procom>/RGBI   -> one combined 4-band GeoTIFF (R, G, B, IR) per tile, already reprojected/ready to crop
input/<procom>/SHP    -> exactly one .shp file: the municipality boundary
```
All outputs for this procom are written under `output/<procom>/`: cropped RGBI
tiles go into `output/<procom>/RGBI`, per-tile index rasters and the final
mosaic are written directly into `output/<procom>/`. The per-tile RGBI crops
and per-tile index rasters are only intermediate products needed to build the
mosaic; the Cleanup section at the end of this notebook deletes them, so that
the only file left under `output/<procom>/` is the final mosaic
(`<city_name>-<procom>-<index>-<flight_year>.tif`) — this is the file every
following step in the pipeline (KDE/KMeans, Australian, Mask+Clump, ...) reads
as its input.</cell id="cell-5">


In [ ]:
red_band = 0
green_band = 1
blue_band = 2
ir_band = 3
crop_orto = True
remote_repository = False
procom = "059033"
city_dict = {'039014': 'Ravenna', 
             '065116': 'Salerno', 
             '031007': 'Gorizia', 
             '038008': 'Ferrara', 
             '051002': 'Arezzo', 
             '048017': 'Firenze',
             '010025': 'Genova',
             '037006': 'Bologna',
             '032006': 'Trieste',
             '015146': 'Milano',
             '063049': 'Napoli',
             '082053': 'Palermo',
             '001272': 'Torino',
             '092009': 'Cagliari',
             '058091': 'Roma',
             '027042': 'Venezia',
             '080063': 'Reggio Calabria',
             '087015': 'Catania',
             '083048': 'Messina',
             '072006': 'Bari',
             '059033': 'Ventotene'}
city_name = city_dict[procom]
indices = ['NDVI_red']  # ['NDVI_red', 'NDVI_blue', 'ENDVI']
flight_year = '2023'
CRScorrespondences = {'TMBOAGA1': 'EPSG:3003', 
                      'TMBOAGA2': 'EPSG:3004', 
                      'WGS 84 / UTM zone 32N': 'EPSG:32632', 
                      'WGS 84 / UTM zone 33N': 'EPSG:32633',
                      'Monte Mario / Italy zone 1': 'EPSG:3003',
                      'RDN2008 / UTM zone 32N': 'EPSG:6707',
                      'RDN2008 / UTM zone 33N': 'EPSG:7792'}

# All inputs/outputs for this procom live under output/<procom>/. Per-tile
# RGBI crops and per-tile index rasters are intermediate products, deleted by
# the Cleanup section at the end of this notebook once the final mosaic has
# been written; only the mosaic is kept for the rest of the pipeline.
base_path = "C:/Users/UTENTE/Downloads/JOS_areeverdi/"
shp_input_path = base_path + "input/" + procom + "/SHP"
rgbi_input_path = base_path + "input/" + procom + "/RGBI"
rgbi_output_path = base_path + "output/" + procom + "/RGBI"
output_path = base_path + "output/" + procom + "/"
indices_output_path = output_path

#create raster images output paths if not exist
if not os.path.exists(rgbi_output_path):
   os.makedirs(rgbi_output_path)
if not os.path.exists(indices_output_path):
   os.makedirs(indices_output_path)

print(procom)
print(city_name)
print(indices)
print(flight_year)
print('shp_input_path:      ', shp_input_path)
print('rgbi_input_path:     ', rgbi_input_path)
print('rgbi_output_path:    ', rgbi_output_path)
print('output_path:         ', output_path)
print('indices_output_path: ', indices_output_path)

059033
Ventotene
['NDVI_red']
2023
shp_input_path:       C:/Users/lancioni/istat/DCME/earth observation/verde urbano/codice_JOS/VerdeUrbanoDaOrtofoto/input/059033/SHP
rgbi_input_path:      C:/Users/lancioni/istat/DCME/earth observation/verde urbano/codice_JOS/VerdeUrbanoDaOrtofoto/input/059033/RGBI
rgbi_output_path:     C:/Users/lancioni/istat/DCME/earth observation/verde urbano/codice_JOS/VerdeUrbanoDaOrtofoto/output/059033/RGBI
output_path:          C:/Users/lancioni/istat/DCME/earth observation/verde urbano/codice_JOS/VerdeUrbanoDaOrtofoto/output/059033/
indices_output_path:  C:/Users/lancioni/istat/DCME/earth observation/verde urbano/codice_JOS/VerdeUrbanoDaOrtofoto/output/059033/


## Functions

Helper functions: vegetation index formulas (`NDVI`, `ENDVI`, `NDVI_vis`, `EVI`), raster I/O, and the crop routine.

In [6]:
def NDVI(nir_value, color_value):
  indx=np.where((1.0*nir_value + 1.0*color_value)>0.0,(1.0*nir_value - 1.0*color_value)/(1.0*nir_value + 1.0*color_value),-20)
  return indx

def ENDVI(nir_value, green_value, blue_value):
  indx=np.where((1.0*nir_value + 1.0*green_value + 2.0*blue_value)>0.0, (1.0*nir_value + 1.0*green_value - 2.0*blue_value) / (1.0*nir_value + 1.0*green_value + 2.0*blue_value), -20)
  return indx

def NDVI_vis(red_value, green_value, blue_value):
  indx=np.where((2.0*green_value + 1.0*red_value + 1.0*blue_value)>0.0, (2.0*green_value - 1.0*red_value - 1.0*blue_value) / (2.0*green_value + 1.0*red_value + 1.0*blue_value), -20)
  return indx

def EVI(nir_value, red_value, blue_value):
  indx=np.where((1.0*nir_value + 6.0*red_value - 7.5*blue_value + 1.0)>0.0, (1.0*nir_value - 1.0*red_value) / (1.0*nir_value + 6.0*red_value - 7.5*blue_value + 1.0), -20)
  return indx

def evaluate_index(rgbi_image, index):
  """
  rgbi_image: single array with 4 bands (R, G, B, IR) as returned by read_image()
  shape of output is [1, x, y] where x, y are the raster dimensions

  band order: red_band=0, green_band=1, blue_band=2, ir_band=3
  """
  x, y = rgbi_image.shape[:2]
  values_array = np.empty([1, x, y])
  ir_image = rgbi_image[:,:,ir_band]
  if index == 'NDVI_red':
    values_array[0] = NDVI(ir_image.astype(np.float32), rgbi_image[:,:,red_band].astype(np.float32))
  elif index == 'NDVI_blue':
    values_array[0] = NDVI(ir_image.astype(np.float32), rgbi_image[:,:,blue_band].astype(np.float32))
  elif index == 'ENDVI':
    values_array[0] = ENDVI(ir_image.astype(np.float32), rgbi_image[:,:,green_band].astype(np.float32), rgbi_image[:,:,blue_band].astype(np.float32))

  return values_array

def read_image(path2filename):
  img = tiff.imread(path2filename)
  img = np.array(img)

  return img

def read_geotiff(filename):
    ds = gdal.Open(filename)
    return ds

# NoData value used for pixels where the index formula's denominator is <= 0
# (see NDVI/ENDVI/NDVI_vis/EVI above). Kept as a single constant so the write
# functions and the mosaic step below all agree on it.
INDEX_NODATA = -20

def write_geotiff_index(filename, arr, in_ds):
    arr_type = gdal.GDT_Float32

    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(filename, arr.shape[1], arr.shape[0], 1, arr_type)
    out_ds.SetProjection(in_ds.GetProjection())
    out_ds.SetGeoTransform(in_ds.GetGeoTransform())

    band = out_ds.GetRasterBand(1)
    band.WriteArray(arr[:,:,0])   # write band to the raster
    band.SetNoDataValue(INDEX_NODATA)  # declare NoData so QGIS/GDAL/rasterio honor it

    out_ds.FlushCache()                     # write to disk
    out_ds = None                           # save, close

def crop_city_raster(vec, raster_input_path, raster_output_dir=None):
    with rasterio.open(raster_input_path) as src:
        Vector=vec.to_crs(src.crs)
        print(Vector.crs)
        out_image, out_transform=mask(src,Vector.geometry,crop=True)
        out_meta=src.meta.copy() # copy the metadata of the source DEM

    out_meta.update({
        "driver":"Gtiff",
        "height":out_image.shape[1], # height starts with shape[1]
        "width":out_image.shape[2], # width starts with shape[2]
        "transform":out_transform
    })

    fileold = os.path.split(raster_input_path)[1]
    filenew = fileold[:-4] + '_loc1.tif'
    out_dir = raster_output_dir if raster_output_dir is not None else os.path.split(raster_input_path)[0]
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    raster_output_path = os.path.join(out_dir, filenew)
    with rasterio.open(raster_output_path,'w',**out_meta) as dst:
        dst.write(out_image)
    
    return(raster_output_path)

def write_compressed_geotiff_index(filename, arr, in_ds):
    arr_type = gdal.GDT_Float32

    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(
        filename,
        arr.shape[1],
        arr.shape[0],
        1,
        arr_type,
        options=[
            "COMPRESS=ZSTD",
            "ZSTD_LEVEL=9",      # opzionale: livello compressione 1-22
            "TILED=YES",         # consigliato per performance
            "BIGTIFF=IF_SAFER"
        ]
    )

    out_ds.SetProjection(in_ds.GetProjection())
    out_ds.SetGeoTransform(in_ds.GetGeoTransform())

    band = out_ds.GetRasterBand(1)
    band.WriteArray(arr[:, :, 0])
    band.SetNoDataValue(INDEX_NODATA)  # declare NoData so QGIS/GDAL/rasterio honor it

    out_ds.FlushCache()
    out_ds = None


In [7]:
i=0
if crop_orto:
    for filename in Path(shp_input_path).glob('*.shp'):
        i=+1
        print(filename)
        loc1=gpd.read_file(filename)
        print(loc1.head())
        print(loc1.crs)
    if i==0:
        print("No shp file in ", shp_input_path)
        assert 1==2
    elif i>1:
        print("Multiple shp files in ", shp_input_path)
        assert 1==2

C:\Users\lancioni\istat\DCME\earth observation\verde urbano\codice_JOS\VerdeUrbanoDaOrtofoto\input\059033\SHP\Ventotene.shp
                                            geometry
0  POLYGON ((874202.055 4525542.892, 874160.897 4...
1  POLYGON ((874032.645 4525617.779, 874027.494 4...
2  POLYGON ((873679.515 4525576.044, 873688.327 4...
3  POLYGON ((873719.808 4525615.673, 873680.891 4...
4  POLYGON ((874180.897 4525638.387, 874152.761 4...
EPSG:32632


## Shapefile

Reads the single `.shp` file expected in `shp_input_path` (the municipality boundary for `procom`) into `loc1`. The cell fails on purpose if the folder has zero or more than one `.shp` file.

In [8]:
loc1.crs

<Projected CRS: EPSG:32632>
Name: WGS 84 / UTM zone 32N
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Between 6°E and 12°E, northern hemisphere between equator and 84°N, onshore and offshore. Algeria. Austria. Cameroon. Denmark. Equatorial Guinea. France. Gabon. Germany. Italy. Libya. Liechtenstein. Monaco. Netherlands. Niger. Nigeria. Norway. Sao Tome and Principe. Svalbard. Sweden. Switzerland. Tunisia. Vatican City State.
- bounds: (6.0, 0.0, 12.0, 84.0)
Coordinate Operation:
- name: UTM zone 32N
- method: Transverse Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [9]:
loc1.head()

,geometry
0,"POLYGON ((874202.055 4525542.892, 874160.897 4..."
1,"POLYGON ((874032.645 4525617.779, 874027.494 4..."
2,"POLYGON ((873679.515 4525576.044, 873688.327 4..."
3,"POLYGON ((873719.808 4525615.673, 873680.891 4..."
4,"POLYGON ((874180.897 4525638.387, 874152.761 4..."


In [ ]:
#retrieve projection and CRS data
print(rgbi_input_path)
#path_to_first_rgbifile = next(os.path.join(rgbi_input_path, f) for f in os.listdir(rgbi_input_path) if os.path.isfile(os.path.join(rgbi_input_path, f)))
path_to_first_rgbifile = next(
    os.path.join(rgbi_input_path, f) 
    for f in os.listdir(rgbi_input_path) 
    if os.path.isfile(os.path.join(rgbi_input_path, f)) and f.lower().endswith(('.tif', '.tiff'))
)
print(path_to_first_rgbifile)
d = read_geotiff(path_to_first_rgbifile)
print(d)
proj = osr.SpatialReference(wkt=d.GetProjection())
commonCRS = CRScorrespondences[str(proj.GetAttrValue('PROJCS', 0))]
print(commonCRS)

C:/Users/lancioni/istat/DCME/earth observation/verde urbano/codice_JOS/VerdeUrbanoDaOrtofoto/input/059033/RGBI
C:/Users/lancioni/istat/DCME/earth observation/verde urbano/codice_JOS/VerdeUrbanoDaOrtofoto/input/059033/RGBI\ventotene.tif
<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x0000029034CDEF10> >
EPSG:7792


## Reference system (CRS)

RGBI input tiles may come in different source coordinate reference systems (e.g. RDN2008 UTM zone 32N/33N, WGS84 UTM zone 32N/33N, Monte Mario). The CRS of the first RGBI file found is read and looked up in `CRScorrespondences` to get the common EPSG code (`commonCRS`), which is then used to reproject the shapefile boundary before cropping.

**If a raster's CRS name is not yet a key of `CRScorrespondences`, add it (with its EPSG code) before running this cell, otherwise a `KeyError` will be raised.**

In [11]:
print(proj)

PROJCS["RDN2008 / UTM zone 33N",
    GEOGCS["RDN2008",
        DATUM["Rete_Dinamica_Nazionale_2008",
            SPHEROID["GRS 1980",6378137,298.257222101,
                AUTHORITY["EPSG","7019"]],
            AUTHORITY["EPSG","1132"]],
        PRIMEM["Greenwich",0,
            AUTHORITY["EPSG","8901"]],
        UNIT["degree",0.0174532925199433,
            AUTHORITY["EPSG","9122"]],
        AUTHORITY["EPSG","6706"]],
    PROJECTION["Transverse_Mercator"],
    PARAMETER["latitude_of_origin",0],
    PARAMETER["central_meridian",15],
    PARAMETER["scale_factor",0.9996],
    PARAMETER["false_easting",500000],
    PARAMETER["false_northing",0],
    UNIT["metre",1,
        AUTHORITY["EPSG","9001"]],
    AXIS["Easting",EAST],
    AXIS["Northing",NORTH],
    AUTHORITY["EPSG","7792"]]


In [12]:
#note: maybe EPSG could be read this way, could replace the CRScorrespondences dictionary?
print(proj.GetAttrValue('AUTHORITY', 0))
print(proj.GetAttrValue('AUTHORITY', 1))

EPSG
7792


## Crop

Crops every RGBI GeoTIFF tile in `rgbi_input_path` to the city boundary (`loc1`), writing the cropped `*_loc1.tif` files into `rgbi_output_path`. Tiles that do not overlap the boundary at all raise a `ValueError` and are skipped.

In [13]:
if crop_orto:
	for filename in Path(rgbi_input_path).glob('*.tif'):
		f = str(filename)
		# checking if it is a file
		if os.path.isfile(f):
			print('tiff path: ', f)
			try:
				n = crop_city_raster(loc1, f, rgbi_output_path)
				print(n)
			except ValueError:
				print("Input shapes do not overlap raster.")
			
			print()

tiff path:  C:\Users\lancioni\istat\DCME\earth observation\verde urbano\codice_JOS\VerdeUrbanoDaOrtofoto\input\059033\RGBI\ventotene.tif
PROJCS["RDN2008 / UTM zone 33N",GEOGCS["RDN2008",DATUM["Rete_Dinamica_Nazionale_2008",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1132"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","6706"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","7792"]]
C:/Users/lancioni/istat/DCME/earth observation/verde urbano/codice_JOS/VerdeUrbanoDaOrtofoto/output/059033/RGBI\ventotene_loc1.tif



## Evaluate indices

For each cropped RGBI tile (`*_loc1.tif` in `rgbi_output_path`), computes the vegetation index/indices listed in `indices` (see `evaluate_index()`) and writes the result as a compressed GeoTIFF into `indices_output_path`.

In [14]:
i=0
for filename in Path(rgbi_output_path).glob('*loc1.tif'):
    if os.path.isfile(filename):
        i=+1
        print(i)
        print(filename)
        rgbi_array = np.asarray(read_image(filename))
        try:
            ds_1 = read_geotiff(str(filename))
            for index in indices:
                indices_values_array = evaluate_index(rgbi_array, index)  # this is where the actual computation happens
                print(indices_values_array.shape)
                write_compressed_geotiff_index(os.path.join(indices_output_path, str(os.path.split(filename)[1][:12]) + '-' + index + '-' + flight_year + '.tif'), np.expand_dims(indices_values_array[0], axis=2), ds_1)  # save as GeoTiff
                del indices_values_array
                gc.collect()
            del ds_1
            gc.collect()
        except ValueError:
            print("operands could not be broadcast together with shape " + str(rgbi_array.shape))
        del rgbi_array
        gc.collect()

        print()

1
C:\Users\lancioni\istat\DCME\earth observation\verde urbano\codice_JOS\VerdeUrbanoDaOrtofoto\output\059033\RGBI\ventotene_loc1.tif


C:\Users\lancioni\AppData\Local\Temp\ipykernel_22596\3937806704.py:2: RuntimeWarning: invalid value encountered in divide
  indx=np.where((1.0*nir_value + 1.0*color_value)>0.0,(1.0*nir_value - 1.0*color_value)/(1.0*nir_value + 1.0*color_value),-20)


(1, 6526, 5309)



## Mosaic

Merges all the per-tile index rasters (`indices_output_path/*<index>-<flight_year>.tif`) into a single city-wide GeoTIFF, saved in `output_path` as `<city_name>-<procom>-<index>-<flight_year>.tif`.

Note: the default overlap strategy is `first` (the first tile written "wins" on overlapping pixels); the result may show visible seams at tile boundaries, and averaging would also need to exclude the `nodata` value (-20) from the mean.</cell id="cell-21">


In [15]:
mosaic_files_written = []  # per-tile index files just merged into the mosaic, for cleanup below

for index in indices:
    search_string = '*' + index + '-' + flight_year + '.tif'
    q = os.path.join(indices_output_path, search_string)
    destination_file = os.path.join(output_path, city_name + '-' + procom + '-' + index + '-' + flight_year + '.tif')

    # Exclude the mosaic's own destination file from the per-tile file list,
    # in case this cell is re-run after a previous mosaic was already written
    # into the same folder.
    original_files = [f for f in glob.glob(q) if os.path.abspath(f) != os.path.abspath(destination_file)]
    print(original_files)
    print(destination_file)

    src_files_to_mosaic = []
    for fp in original_files:
        src = rasterio.open(fp)
        src_files_to_mosaic.append(src)

    try:
        mosaic, out_trans = merge(src_files_to_mosaic, nodata=INDEX_NODATA)
        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": out_trans,
            # output dtype
            "dtype": mosaic.dtype,
            # NoData: declare it explicitly rather than relying on it being
            # inherited from the last source file's metadata
            "nodata": INDEX_NODATA,
            # compression
            "compress": "ZSTD",
            "zstd_level": 9,
            # recommended for float rasters
            "predictor": 2,
            # tiled geotiff
            "tiled": True,
            # optional
            "BIGTIFF": "IF_SAFER"
        })
        # Close the source file handles before deleting them in the Cleanup
        # section below (Windows keeps an open file locked).
        for src_fh in src_files_to_mosaic:
            src_fh.close()

        with rasterio.open(destination_file, "w", **out_meta) as dest:
            dest.write(mosaic)

        # Only track these per-tile files for cleanup once the mosaic that
        # depends on them has actually been written successfully.
        mosaic_files_written.extend(original_files)

    except MemoryError:
        print('Mosaic too large for dtype=np.float32; trying np.float16')
        for src_fh in src_files_to_mosaic:
            src_fh.close()
        mosaic, out_trans = merge(src_files_to_mosaic, nodata=INDEX_NODATA, dtype=np.float16)
        # GeoTIFF does not support float16
        np.save(destination_file, mosaic)
        mosaic_files_written.extend(original_files)

    except Exception as e:
        print(f'Something went wrong: {e}')
        for src_fh in src_files_to_mosaic:
            src_fh.close()

['C:/Users/lancioni/istat/DCME/earth observation/verde urbano/codice_JOS/VerdeUrbanoDaOrtofoto/output/059033\\ventotene_lo-NDVI_red-2023.tif']
C:/Users/lancioni/istat/DCME/earth observation/verde urbano/codice_JOS/VerdeUrbanoDaOrtofoto/output/059033/Ventotene-059033-NDVI_red-2023.tif


## Cleanup

This notebook is meant to produce a single deliverable: the final city-wide
mosaic (`output/<procom>/<city_name>-<procom>-<index>-<flight_year>.tif`),
which every following step of the pipeline reads as its input. Everything
else written above -- the cropped RGBI tiles (`rgbi_output_path`) and the
per-tile index rasters (the ones just merged into the mosaic) -- is only an
intermediate product, so it is deleted here:

- all cropped RGBI tiles (`*_loc1.tif`) in `rgbi_output_path`, and the folder itself if it ends up empty;
- the per-tile index rasters that were successfully merged into the mosaic above (tracked in `mosaic_files_written`).

The mosaic file itself is never touched. If the mosaic step above failed or
was skipped for a given index (e.g. `MemoryError`), the per-tile files for
that index are intentionally left in place instead of being deleted, so no
data is lost.

In [16]:
# Delete every cropped RGBI tile (intermediate product of the Crop step)
deleted_rgbi = 0
for f in Path(rgbi_output_path).glob('*_loc1.tif'):
    try:
        os.remove(f)
        deleted_rgbi += 1
    except OSError as e:
        print(f'Could not delete {f}: {e}')
print('Deleted cropped RGBI tiles: ', deleted_rgbi)

# Remove the RGBI output folder itself if it is now empty
try:
    if os.path.isdir(rgbi_output_path) and not os.listdir(rgbi_output_path):
        os.rmdir(rgbi_output_path)
        print('Removed empty folder: ', rgbi_output_path)
except OSError as e:
    print(f'Could not remove folder {rgbi_output_path}: {e}')

# Delete only the per-tile index rasters that were actually merged into a
# mosaic above (mosaic_files_written); anything left out of that list (e.g.
# because the mosaic step failed for that index) is intentionally kept.
deleted_index_tiles = 0
for f in mosaic_files_written:
    try:
        os.remove(f)
        deleted_index_tiles += 1
    except OSError as e:
        print(f'Could not delete {f}: {e}')
print('Deleted per-tile index rasters: ', deleted_index_tiles)

print()
print('Remaining files in output_path:')
for f in sorted(os.listdir(output_path)):
    print(' -', f)

Deleted cropped RGBI tiles:  1
Removed empty folder:  C:/Users/lancioni/istat/DCME/earth observation/verde urbano/codice_JOS/VerdeUrbanoDaOrtofoto/output/059033/RGBI
Deleted per-tile index rasters:  1

Remaining files in output_path:
 - Ventotene-059033-NDVI_red-2023.tif
